In [ ]:
import re
from pathlib import Path

def parse_morphological_features(features_str):
    """
    Parse morphological features from CoNLL-U format into UniMorph-style tags.
    Converts Georgian verb features like Number[subj]=Sing|Person[subj]=3 
    into NOM(3,SG) format.
    """
    if not features_str or features_str == '_':
        return 'V'
    
    features = features_str.split('|')
    feature_dict = {}
    
    # Parse all features into a dictionary
    for feature in features:
        if '=' not in feature:
            continue
        key, value = feature.split('=', 1)
        feature_dict[key] = value
    
    # Start building tag components
    tags = ['V']
    
    # Mood
    mood_map = {
        'Ind': 'IND',
        'Sub': 'SBJV',
        'Imp': 'IMP',
        'Cnd': 'COND',
        'Opt': 'OPT'
    }
    if 'Mood' in feature_dict:
        mood = mood_map.get(feature_dict['Mood'], feature_dict['Mood'].upper())
        tags.append(mood)
    
    # Tense/Aspect (combine if both present)
    tense_map = {
        'Pres': 'PRS',
        'Past': 'PST',
        'Fut': 'FUT',
        'Imp': 'IPFV',
        'Aor': 'PST'  # Aorist treated as past
    }
    aspect_map = {
        'Perf': 'PRF',
        'Imp': 'IPFV',
        'Prog': 'PROG'
    }
    
    if 'Tense' in feature_dict:
        tense = tense_map.get(feature_dict['Tense'], feature_dict['Tense'].upper())
        tags.append(tense)
    elif 'Aspect' in feature_dict:
        aspect = aspect_map.get(feature_dict['Aspect'], feature_dict['Aspect'].upper())
        tags.append(aspect)
    
    # Voice (if present and not redundant)
    voice_map = {
        'Act': 'ACT',
        'Pass': 'PASS',
        'Mid': 'MID',
        'MedPass': 'MID'
    }
    if 'Voice' in feature_dict and feature_dict['Voice'] in voice_map:
        tags.append(voice_map[feature_dict['Voice']])
    
    # Subject (NOM) - from Number[subj] and Person[subj]
    person_subj = feature_dict.get('Person[subj]', feature_dict.get('Person'))
    number_subj = feature_dict.get('Number[subj]', feature_dict.get('Number'))
    
    if person_subj and number_subj:
        num = 'SG' if number_subj == 'Sing' else 'PL'
        tags.append(f"NOM({person_subj},{num})")
    
    # Object (ACC) - from Number[obj] and Person[obj]
    person_obj = feature_dict.get('Person[obj]')
    number_obj = feature_dict.get('Number[obj]')
    
    if person_obj and number_obj:
        num = 'SG' if number_obj == 'Sing' else 'PL'
        tags.append(f"ACC({person_obj},{num})")
    
    # Indirect Object (DAT) - from Number[io] and Person[io]
    person_io = feature_dict.get('Person[io]')
    number_io = feature_dict.get('Number[io]')
    
    if person_io and number_io:
        num = 'SG' if number_io == 'Sing' else 'PL'
        tags.append(f"DAT({person_io},{num})")
    
    # Polarity (negation)
    if 'Polarity' in feature_dict and feature_dict['Polarity'] == 'Neg':
        tags.append('NEG')
    
    # VerbForm (if non-finite)
    verbform_map = {
        'Inf': 'INF',
        'Part': 'PTCP',
        'Ger': 'GER',
        'Vnoun': 'VN'
    }
    if 'VerbForm' in feature_dict and feature_dict['VerbForm'] in verbform_map:
        vf = verbform_map[feature_dict['VerbForm']]
        if vf != 'Fin':  # Don't add finite marker
            tags.append(vf)
    
    return ';'.join(tags)


def process_conllu_sentence(lines):
    """
    Process a single CoNLL-U sentence block.
    Returns tuple: (lemma, tags, form, context) or None if no verb found.
    """
    context = None
    verb_data = None
    
    for line in lines:
        line = line.strip()
        
        # Extract context from text comment
        if line.startswith('# text = '):
            context = line[9:].strip()
            continue
        
        # Skip comments and empty lines
        if line.startswith('#') or not line:
            continue
        
        # Parse token line
        parts = line.split('\t')
        if len(parts) < 6:
            continue
        
        token_id = parts[0]
        # Skip multiword tokens (e.g., "1-2")
        if '-' in token_id or '.' in token_id:
            continue
        
        form = parts[1]        # Inflected form
        lemma = parts[2]       # Lemma (infinitive)
        upos = parts[3]        # Universal POS tag
        feats = parts[5]       # Morphological features
        
        # Check if this is a verb (first one we encounter)
        if upos == 'VERB' and verb_data is None:
            tags = parse_morphological_features(feats)
            verb_data = (lemma, tags, form)
    
    # Return the verb data with context if both were found
    if verb_data and context:
        return verb_data + (context,)
    return None


def convert_conllu_to_unimorph(input_file, output_file):
    """
    Convert a CoNLL-U file to unimorph format with context.
    """
    sentence_lines = []
    results = []
    
    with open(input_file, 'r', encoding='utf-8') as fin:
        for line in fin:
            line = line.rstrip('\n')
            
            # Empty line marks end of sentence
            if not line:
                if sentence_lines:
                    result = process_conllu_sentence(sentence_lines)
                    if result:
                        results.append(result)
                    sentence_lines = []
            else:
                sentence_lines.append(line)
        
        # Process last sentence if file doesn't end with empty line
        if sentence_lines:
            result = process_conllu_sentence(sentence_lines)
            if result:
                results.append(result)
    
    # Write output
    with open(output_file, 'w', encoding='utf-8') as fout:
        for lemma, tags, form, context in results:
            fout.write(f"{lemma}\t{tags}\t{form}\t{context}\n")
    
    return len(results)


## Test with a sample

In [2]:
# Test with sample data
sample_conllu = """# sent_id = 1
# text = Aquele cliente gosta apenas de vinho branco .
1	Aquele	aquele	DET	DEM	Gender=Masc|Number=Sing|PronType=Dem	2	det	_	_
2	cliente	cliente	NOUN	CN	Gender=Masc|Number=Sing	3	nsubj	_	_
3	gosta	gostar	VERB	V	Mood=Ind|Number=Sing|Person=3|Tense=Pres|VerbForm=Fin	0	root	_	_
4	apenas	apenas	ADV	ADV	_	6	advmod	_	_
5	de	de	ADP	PREP	_	6	case	_	_
6	vinho	vinho	NOUN	CN	Gender=Masc|Number=Sing	3	obl	_	_
7	branco	branco	ADJ	ADJ	Gender=Masc|Number=Sing	6	amod	_	_
8	.	.	PUNCT	PNT	_	3	punct	_	_

"""

# Write sample to file
with open('sample.conllu', 'w', encoding='utf-8') as f:
    f.write(sample_conllu)

# Process it
lines = sample_conllu.strip().split('\n')
result = process_conllu_sentence(lines)

if result:
    lemma, tags, form, context = result
    print("Sample output:")
    print(f"{lemma}\t{tags}\t{form}\t{context}")
else:
    print("No verb found in sample")

Sample output:
gostar	V;IND;SG;3;PRS	gosta	Aquele cliente gosta apenas de vinho branco .


## Process all CoNLL-U files from a language

In [ ]:
# Configuration for different languages
# Add new languages here with their CoNLL-U file prefixes and output codes

LANGUAGE_CONFIGS = {
    'georgian': {
        'prefix': 'ka_gnc-ud',
        'code': 'kat',
        'name': 'Georgian'
    },
    'portuguese': {
        'prefix': 'pt_bosque-ud',
        'code': 'por',
        'name': 'Portuguese'
    },
    'english': {
        'prefix': 'en_ewt-ud',
        'code': 'eng',
        'name': 'English'
    },
    'italian': {
        'prefix': 'it_isdt-ud',
        'code': 'ita',
        'name': 'Italian'
    },
    'greek': {
        'prefix': 'grc_proiel-ud',
        'code': 'grc',
        'name': 'Ancient Greek'
    }
}

# Select language to process (change this to process different languages)
CURRENT_LANGUAGE = 'georgian'

config = LANGUAGE_CONFIGS[CURRENT_LANGUAGE]
files_to_process = [
    (f"{config['prefix']}-train.conllu", f"{config['code']}.trn"),
    (f"{config['prefix']}-dev.conllu", f"{config['code']}.dev"),
    (f"{config['prefix']}-test.conllu", f"{config['code']}.tst")
]

print(f"Processing {config['name']} data...\n")

for input_file, output_file in files_to_process:
    if Path(input_file).exists():
        count = convert_conllu_to_unimorph(input_file, output_file)
        print(f"✓ {input_file}: {count} sentences -> {output_file}")
    else:
        print(f"✗ {input_file}: not found")


✓ ka_gnc-ud-train.conllu: 798 sentences -> kat.trn
✓ ka_gnc-ud-dev.conllu: 120 sentences -> kat.dev
✓ ka_gnc-ud-test.conllu: 741 sentences -> kat.tst


## Sample output inspection

Let's check a few examples to verify the feature extraction:

In [ ]:
# Read and display first 5 lines of output to verify
output_file = f"{config['code']}_verbs_context.trn"
if Path(output_file).exists():
    print(f"First 5 examples from {output_file}:\n")
    with open(output_file, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            parts = line.strip().split('\t')
            if len(parts) == 4:
                lemma, tags, form, context = parts
                print(f"Example {i+1}:")
                print(f"  Lemma: {lemma}")
                print(f"  Tags:  {tags}")
                print(f"  Form:  {form}")
                print(f"  Context: {context[:80]}..." if len(context) > 80 else f"  Context: {context}")
                print()
else:
    print(f"Output file {output_file} not found. Run the processing cell first.")
